# Laufzeitspezifische Initialisierung & Tests (hier: Google Colab mit Google Drive)

# Einleitung

Faster R-CNN Trainingspipeline für Baumarterkennung auf Orthofotos (Münster Zentrum)
================================================================================

## Aufbau des Notebooks:

Zunächst werden...
- Parameter gesetzt
- Funktionenen definiert, die die Daten aufbereiten (hier im linearen Skript; alternativ ausgelagert in Unterordner 'src'):
  - load_tile_local,
  - load_trees_local,
  - unify_crs, trees_geo_to_pixel,
  - tree_selection,
  - create_boxes,
  - plot_boxes_on_tile
  - remove_non_trees_safe   (Geometrie-basierter Merge statt Positions-Matching)
  - Gruppierung seltener Baumarten in "Sonstige"
  - Patch-Erzeugung aus grossen (10000x10000 px) Kacheln
  - TreePatchDataset (torch Dataset)
  - Predictor-Umbau auf eigene Klassenzahl
  - Trainingsloop

Dann wird weiter unten das Modell definiert und trainiert: Code chunk  'Modell & Training' mit den den Funktionen build_model und train_model

Dann wird das Modell evaluiert: Code Chunk 'Auswertung: Loss-Kurve, mAP auf Testset, visuelle Kontrolle'

## Man beachte
Implementierung für Laufzeit Google Colab.

Subfolder DATA sollte die gewünschten Luftbilder enthalten (im georeferenzierten Format .jp2; Anzahl in Abschnitt Konfiguration bei ntiles anpassen!).

Tuning des Workflows:

PLATZHALTER (pruefen):
    - BOX_WIDTH: Platzhalter 160 (visuell mit plot_boxes_on_tile pruefen)
    - RARE_SPECIES_THRESHOLD: Platzhalter 100 (anhand value_counts() anpassen)
    - remove_non_trees_safe: Abgleich gruen_opendata.geojson (ungefiltertes Baumkataster) und baeume.gpkg (gefiltertes Baumkataster) sinnvoll, ob die Zeilen die gleiche Reihenfolge haben (kein expliziter Zeilenindex als Variable verfügbar)

# Konfiguration

## Laufzeispezifische Konfiguration (hier: Google Colab)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os

print("Current working directory:", os.getcwd())
print("Contents of current directory:", os.listdir('.'))

Current working directory: /content
Contents of current directory: ['.config', 'sample_data']


In [3]:
# Change the current working directory to PROJECT_ROOT_DIR
# This is crucial for relative imports and package discovery in notebooks.
# Using %cd (a Colab magic command) is often more reliable than os.chdir() for persistence.
%cd /content/drive/MyDrive/techlabs_trees_26

import os
import sys
from pathlib import Path

# --- IMPORTANT: Configure your project root directory on Google Drive ---
# The directory on Google Drive that contains your 'src' and 'DATA' folders.
# Example: PROJECT_ROOT_DIR = '/content/drive/MyDrive/MyProjectName'
# Please replace 'YOUR_PROJECT_ROOT_PATH' with the actual path.
PROJECT_ROOT_DIR = '/content/drive/MyDrive/techlabs_trees_26'

# Ensure PROJECT_ROOT_DIR is at the beginning of sys.path
if PROJECT_ROOT_DIR in sys.path:
    sys.path.remove(PROJECT_ROOT_DIR)
sys.path.insert(0, PROJECT_ROOT_DIR)

# Explicitly add the 'src' directory to sys.path to ensure modules within it are discoverable.
SRC_DIR = os.path.join(PROJECT_ROOT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)


print(f"Current working directory after %cd: {os.getcwd()}")
# --- Diagnostic additions ---
print(f"PROJECT_ROOT_DIR set to: {PROJECT_ROOT_DIR}")
if not os.path.exists(PROJECT_ROOT_DIR):
    print(f"WARNING: PROJECT_ROOT_DIR does not exist: {PROJECT_ROOT_DIR}")
else:
    print(f"Contents of PROJECT_ROOT_DIR: {os.listdir(PROJECT_ROOT_DIR)}")
    if not os.path.exists(os.path.join(PROJECT_ROOT_DIR, 'src')):
        print(f"ERROR: 'src' directory not found in {PROJECT_ROOT_DIR}. Please ensure 'src' is a direct subdirectory of PROJECT_ROOT_DIR.")
    else:
        print(f"'src' directory found in {PROJECT_ROOT_DIR}. Listing contents:")
        print(os.listdir(os.path.join(PROJECT_ROOT_DIR, 'src')))
print(f"Current sys.path: {sys.path}") # Added to verify sys.path contents
# --- End diagnostic additions ---


# If DATA_PATH (defined later) uses Path(".. DATA"), and your notebook is
# in a subfolder (e.g., 'notebooks') within PROJECT_ROOT_DIR,
# you might need to adjust os.chdir or DATA_PATH definition accordingly.
# For example, if notebook is in MyProject/notebooks, and DATA is in MyProject/DATA:
# os.chdir(os.path.join(PROJECT_ROOT_DIR, 'notebooks'))
# Then DATA_PATH = Path(".. DATA") would work.
# If your notebook is directly in PROJECT_ROOT_DIR, then DATA_PATH should be Path("DATA").
# Consider if you need to change the current working directory (os.chdir)
# to where your notebook actually resides within the project for relative paths to work.



/content/drive/MyDrive/techlabs_trees_26
Current working directory after %cd: /content/drive/MyDrive/techlabs_trees_26
PROJECT_ROOT_DIR set to: /content/drive/MyDrive/techlabs_trees_26
Contents of PROJECT_ROOT_DIR: ['DL_Gr2_2026SoSe_abgabe.ipynb', '.git', '.gitignore', 'README.md', 'howto_and_best_practice.md', 'meeting_notes', 'src', 'DATA', 'baumerkennung_straßenpuffer_phase-a.ipynb', 'DrawingBoxes.ipynb']
'src' directory found in /content/drive/MyDrive/techlabs_trees_26. Listing contents:
['__init__.py', 'box_creation.py', 'convert_crs.py', 'load_data.py', 'tree_selection.py', '__pycache__']
Current sys.path: ['/content/drive/MyDrive/techlabs_trees_26/src', '/content/drive/MyDrive/techlabs_trees_26', '/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload', '', '/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.13/dist-packages/IPython/extensions', '/root/.ipython']


## Import von python und pytorch packages

In [4]:
import random

import geopandas as gpd
import numpy as np
import torch
from rasterio.windows import Window
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# für die Evaluations-Funktion (ganz unten)
try:
    import torchmetrics
except ImportError:
    !pip install torchmetrics -q

# Hier ehemals in PY-Skripte ausgelagerte Funktions-Definitionen auskommentiert, da 'src' als subfolder nicht lesbar war;
# # ... alle Funktions-Defintionen hier im Skript.

#from src.load_data import load_tile_local, load_trees_local
#from src.convert_crs import unify_crs, trees_geo_to_pixel
#from src.tree_selection import tree_selection
#from src.box_creation import plot_boxes_on_tile  # nur zur optionalen Visualisierung

print("All imports attempted.")

All imports attempted.


## Parameter setzen

In [36]:
# ---------------------------------------------------------------------------
# 1. Konfiguration
# ---------------------------------------------------------------------------

# Construct absolute paths using PROJECT_ROOT_DIR
DATA_PATH = Path(PROJECT_ROOT_DIR) / "DATA"
NTILES = 4
FILE_EXT = ".jp2"
FILTERED_TREE_PATH = DATA_PATH / "baeume.gpkg"

PATCH_SIZE = 1024          # Pixelgroesse der Patches (quadratisch)
PATCH_STRIDE = 1024        # kein Overlap
BOX_WIDTH = 160            # PLATZHALTER - visuell pruefen
RARE_SPECIES_THRESHOLD = 100  # PLATZHALTER - anhand value_counts() anpassen

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
# TEST_FRAC ergibt sich als Rest

BATCH_SIZE = 2
NUM_EPOCHS = 5              # didaktischer Umfang, bewusst klein gehalten
LEARNING_RATE = 0.005
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Datenvorbereitung

## Funktionen: Definitionen
Nachfolgend werden einige Funktionen zum Laden und Aufbereiten der Daten definiert.

### Funktion-Def: load_tile_local
Hiermit werden beim späteren Aufruf der Funktion die Luftbilddaten mit der library rasterio geladen.

In [6]:
# # # # # # # # # load data of tiles # # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def load_tile_local(data_path, ntiles, file_extension):
    # import rasterio
    import os
    import rasterio
    # get a list of all files of the requested format.
    # Put them in a list called tilenames
    filenames = os.listdir(data_path)
    tilenames = [
        data_path / filename
        for filename in filenames
        if filename.endswith(file_extension)
    ]
    # open ntiles tiles
    tiles = []
    for tilename in tilenames[0:ntiles]:
        data = rasterio.open(tilename)
        tiles.append({
            "name": tilename.name,
            "data": data
        })
    return tiles

#                                                            #
# #                                                        # #
# # # # # # # # # end load data of tiles # # # # # # # # # # #

### Funktion-Def: load_trees_local
Hier werden die Baumkatasterdaten geladen

In [7]:
# # # # # # # # # load data of trees # # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def load_trees_local(data_path):
    # import os, geopandas
    import os
    import geopandas as gpd
    # get a list of all files in the folder.
    # Select corresponding file based on file extension
    filenames = os.listdir(data_path)
    tree_file = [
        data_path / filename
        for filename in filenames
        if filename.endswith("opendata.geojson")
    ]

    tree_file = tree_file[0]

    # open geojson file using geopandas
    trees = gpd.read_file(tree_file)
    return trees

#                                                            #
# #                                                        # #
# # # # # # # # # end load data of trees # # # # # # # # # # #

### Funktion-Def: Geokoordinaten transformieren
Hiermit werden die Geokoordinaten des Baumkatasters zunächst an das Referenzsystem der Orthophotos (tiles) angepasst (unify_crs); und dann in Pixel-Koordinaten der Tiles umgerechnet (trees_geo_to_pixel).

In [8]:
# # # # # # convert crs of trees to aerial data  # # # # # # #
# #                                                        # #
#                                                            #
def unify_crs(tiles, trees):
    # get crs of tiles
    for tile in tiles:
        tiledata = tile["data"]
        tile_crs = tiledata.crs

    # get crs of trees
    tree_crs = trees.crs.to_epsg()

    # unify if not the same already
    if tree_crs != tile_crs:
        trees = trees.to_crs(tile_crs)

    # return
    return trees

#                                                            #
# #                                                        # #
# # # # # end convert crs of trees to aerial data  # # # # # #


# # # # # # convert geodata of trees to pixel  # # # # # # # #
# #                                                        # #
#                                                            #
def trees_geo_to_pixel(tiles, trees, target_tile):
    from rasterio.transform import rowcol
    pixels = []
    tile   = tiles[target_tile]["data"]
    for point in trees.geometry:
        row, col = rowcol(
            tile.transform,
            point.x,
            point.y
        )
        pixels.append(
            (col, row)
        )

    trees['pixels'] = pixels

    trees["pixel_x"] = trees["pixels"].apply(lambda p: p[0])
    trees["pixel_y"] = trees["pixels"].apply(lambda p: p[1])

    return trees

#                                                            #
# #                                                        # #
# # # # # end convert crs of trees to aerial data  # # # # # #


### Funktion-Def: Bäume innerhalb der tiles auswählen
Hier wird das Münsteraner Baumkataster auf diejenigen Fälle von Bäumen reduziert, die auch auf den betrachteten Luftbildern (tiles) enthalten sind (tree_selection)

In [41]:
# # # # # # select trees that are within a tile  # # # # # # #
# #                                                        # #
#                                                            #
def tree_selection(tiles, trees, target_tile):
    tile = tiles[target_tile]["data"]
    max_pix_x = tile.width
    max_pix_y = tile.height
    logical_index = (
        (trees["pixel_x"] >= 0) &
        (trees["pixel_y"] >= 0) &
        (trees["pixel_x"] < max_pix_x) &
        (trees["pixel_y"] < max_pix_y)
    )
    trees_within_tile = trees[logical_index]

    return trees_within_tile

#                                                            #
# #                                                        # #
# # # # # end tree selection # # # # # # # # # # # # # # # # #

In [10]:
# ---------------------------------------------------------------------------
# Robuster Ersatz fuer remove_non_trees (Geometrie-Merge statt Positions-Matching)
# ---------------------------------------------------------------------------

# # # # # # remove non-tree entries (SAFE, Geometrie-Matching) # # # # # #
# #                                                        # #
#                                                            #
def remove_non_trees_safe(file_path, trees):
    """
    Robusterer Ersatz fuer remove_non_trees(). Verknuepft trees und
    trees_filtered ueber gerundete Koordinaten statt ueber Zeilenreihenfolge/
    Index, da keine gemeinsame ID-Spalte existiert.

    ACHTUNG: noch nicht gegen die echten Daten verifiziert (siehe
    Projektnotizen). Vor produktivem Einsatz Stichprobenvergleich
    durchfuehren.
    """
    import geopandas as gpd
    trees_filtered = gpd.read_file(file_path)

    trees = trees.copy()
    trees_filtered = trees_filtered.copy()

    # WICHTIG: trees_filtered auf dasselbe CRS wie trees bringen, bevor
    # geom_key gebildet wird. Ohne diesen Schritt werden Grad- gegen
    # Meterkoordinaten verglichen, was zu massenhaften Kollisionen fuehrt.
    if trees_filtered.crs != trees.crs:
        trees_filtered = trees_filtered.to_crs(trees.crs)

    # Rundung: bei projiziertem CRS (Meter) ist 1 Dezimalstelle (=10cm)
    # praezise genug, um verschiedene Baeume zu unterscheiden, aber tolerant
    # genug gegenueber minimalen numerischen Abweichungen zwischen den
    # beiden Exporten. Bei Bedarf anpassen.
    ROUND_DECIMALS = 1

    trees["geom_key"] = trees.geometry.apply(
        lambda g: (round(g.x, ROUND_DECIMALS), round(g.y, ROUND_DECIMALS))
    )
    trees_filtered["geom_key"] = trees_filtered.geometry.apply(
        lambda g: (round(g.x, ROUND_DECIMALS), round(g.y, ROUND_DECIMALS))
    )

    # Sicherheitscheck: verbleibende Duplikate innerhalb trees_filtered
    # nach korrektem CRS-Angleich waeren ein Hinweis auf tatsaechlich
    # identische/sehr nah beieinanderliegende Baumkoordinaten (z.B. zwei
    # Baeume derselben Allee), nicht mehr auf ein CRS-Mismatch.
    dup_mask = trees_filtered["geom_key"].duplicated(keep=False)
    dup_count = dup_mask.sum()
    if dup_count > 0:
        print(
            f"WARNUNG: {dup_count} Zeilen mit doppeltem geom_key in "
            f"trees_filtered nach CRS-Angleich (Rundung {ROUND_DECIMALS} "
            f"Nachkommastellen). Betroffene Zeilen:"
        )
        print(trees_filtered.loc[dup_mask, ["geom_key"]])
        # Erste Zeile pro geom_key behalten, Rest verwerfen. Bei nur
        # wenigen betroffenen Zeilen (z.B. dicht beieinanderstehende
        # Baeume) ein vertretbarer Kompromiss - bei vielen Faellen sollte
        # stattdessen ROUND_DECIMALS erhoeht werden, um echte Baeume nicht
        # zusammenzufassen.
        trees_filtered = trees_filtered.drop_duplicates(subset="geom_key", keep="first")

    dup_mask_left = trees["geom_key"].duplicated(keep=False)
    if dup_mask_left.sum() > 0:
        print(
            f"WARNUNG: {dup_mask_left.sum()} Zeilen mit doppeltem geom_key "
            f"in trees (Original-Baumdaten). Betroffene Zeilen:"
        )
        print(trees.loc[dup_mask_left, ["geom_key"]])

    n_before = len(trees)
    trees = trees.merge(
        trees_filtered[["geom_key", "is_True"]],
        on="geom_key",
        how="left",
    )
    n_after = len(trees)
    if n_after != n_before:
        print(
            f"WARNUNG: Zeilenanzahl hat sich beim Merge veraendert "
            f"({n_before} -> {n_after}). Das deutet auf verbleibende "
            f"Mehrfachtreffer hin - Ergebnis pruefen, bevor weitergemacht wird."
        )

    trees = trees[trees["is_True"] == 1]
    return trees

#                                                            #
# #                                                        # #
# # # # # end non_trees_safe  # # # # # # # # # # # # # # # #


# # # # # # group rare species into "Sonstige"  # # # # # # #
# #                                                        # #
#                                                            #
def group_rare_species(trees, species_col="baumgruppe", threshold=100):
    """
    threshold: PLATZHALTER - anhand der vollstaendigen value_counts()-
    Verteilung anpassen (bisher nicht final festgelegt).
    """
    counts = trees[species_col].value_counts()
    rare_species = counts[counts < threshold].index

    trees = trees.copy()
    trees[f"{species_col}_grouped"] = trees[species_col].where(
        ~trees[species_col].isin(rare_species), "Sonstige"
    )
    return trees

#                                                            #
# #                                                        # #
# # # # # end group rare species # # # # # # # # # # # # # # #


In [11]:

# ---------------------------------------------------------------------------
# 3. Seltene Baumarten zu "Sonstige" zusammenfassen
# ---------------------------------------------------------------------------

# # # # # # group rare species into "Sonstige"  # # # # # # #
# #                                                        # #
#                                                            #
def group_rare_species(trees, species_col="baumgruppe", threshold=100):
    """
    threshold: PLATZHALTER - anhand der vollstaendigen value_counts()-
    Verteilung anpassen (bisher nicht final festgelegt).
    """
    counts = trees[species_col].value_counts()
    rare_species = counts[counts < threshold].index

    trees = trees.copy()
    trees[f"{species_col}_grouped"] = trees[species_col].where(
        ~trees[species_col].isin(rare_species), "Sonstige"
    )
    return trees

#

### Funktion-Def: Von Tiles zu Patches
Für das Modelltraining werden nicht ganze tiles verwendet, sondern tiles werden in kleinere Entitäten, sogenannte 'batches' zerlegt (siehe Anzahl und Größe der batches auch in Abschnitt Konfiguration; ein tile geht dabei nich zu 100% in batches auf).

In [12]:
# ---------------------------------------------------------------------------
# 4. Patch-Fenster pro Kachel erzeugen
# ---------------------------------------------------------------------------

def generate_patch_windows(tile_width, tile_height, patch_size=PATCH_SIZE, stride=PATCH_STRIDE):
    """
    Liefert Liste von (col_off, row_off). Randstaendige, unvollstaendige
    Patches werden verworfen (einfachste Variante, kein Padding).
    """
    windows = []
    for row_off in range(0, tile_height - patch_size + 1, stride):
        for col_off in range(0, tile_width - patch_size + 1, stride):
            windows.append((col_off, row_off))
    return windows

### Funktion-Def: Bounding Boxes für Faster R-CNN erstellen
Mit der Funktion create_boxes_patch werden die Bounding Boxes erstellt, die Faster R-CNN für das Modelltraining benötigt. Diese Bounding Boxes ergeben sich als Flächen um die Punktkoordinaten des Baumkatasters. Bzgl. box_width sieh Abschnitt Konfiguration/ Parameter setzen.

In [13]:
# ---------------------------------------------------------------------------
# 5. Boxen relativ zu einem Patch berechnen (Patch-Variante von create_boxes)
# ---------------------------------------------------------------------------

def create_boxes_patch(trees_within_tile, col_off, row_off, patch_size, box_width):
    half_box = box_width / 2
    boxes = []
    valid_indices = []
    for idx, row in trees_within_tile.iterrows():
        x = row["pixel_x"] - col_off
        y = row["pixel_y"] - row_off
        xmin, ymin = x - half_box, y - half_box
        xmax, ymax = x + half_box, y + half_box
        if xmin >= 0 and ymin >= 0 and xmax < patch_size and ymax < patch_size:
            boxes.append([xmin, ymin, xmax, ymax])
            valid_indices.append(idx)
    return boxes, valid_indices


### Datenset das Modelltraining aufbereiten

In [14]:
# ---------------------------------------------------------------------------
# 6. Dataset-Klasse
# ---------------------------------------------------------------------------

class TreePatchDataset(Dataset):
    """
    patch_index: Liste von (tile_idx, col_off, row_off)
    trees_by_tile: dict tile_idx -> trees_within_tile
                   (bereits durch trees_geo_to_pixel + tree_selection vorbereitet,
                    enthaelt pixel_x, pixel_y, baumgruppe_grouped)
    """

    def __init__(self, tiles, trees_by_tile, patch_index, label_map,
                 patch_size=PATCH_SIZE, box_width=BOX_WIDTH,
                 label_col="baumgruppe_grouped"):
        self.tiles = tiles
        self.trees_by_tile = trees_by_tile
        self.patch_index = patch_index
        self.label_map = label_map
        self.patch_size = patch_size
        self.box_width = box_width
        self.label_col = label_col

    def __len__(self):
        return len(self.patch_index)

    def __getitem__(self, idx):
        tile_idx, col_off, row_off = self.patch_index[idx]
        dataset = self.tiles[tile_idx]["data"]

        window = Window(col_off, row_off, self.patch_size, self.patch_size)
        img_array = dataset.read(window=window)  # (bands, patch_size, patch_size)
        image = torch.from_numpy(img_array[:3].astype(np.float32)) / 255.0

        trees = self.trees_by_tile[tile_idx]
        boxes, valid_indices = create_boxes_patch(
            trees, col_off, row_off, self.patch_size, self.box_width
        )

        if boxes:
            boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
            labels_t = torch.as_tensor(
                trees.loc[valid_indices, self.label_col].map(self.label_map).values,
                dtype=torch.int64,
            )
        else:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)

        target = {"boxes": boxes_t, "labels": labels_t}
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [21]:
# # # # # # # # build_dataset_components # # # # # # # # # # #
# #                                                        # #
#                                                            #
def build_dataset_components(data_path, ntiles, file_ext, filtered_tree_path,
                              rare_species_threshold=100,
                              patch_size=1024, patch_stride=1024):
    """
    Fuehrt die gesamte Vorverarbeitungskette aus:
    laden -> CRS vereinheitlichen -> Nicht-Baeume entfernen ->
    seltene Arten gruppieren -> pro Kachel Pixelkoordinaten berechnen
    und filtern -> Patch-Fenster erzeugen.

    data_path: Path-Objekt zum DATA-Verzeichnis
    filtered_tree_path: Pfad zur .gpkg-Datei mit is_True-Markierung
    """
    tiles = load_tile_local(data_path, ntiles, file_ext)
    trees = load_trees_local(data_path)
    trees = unify_crs(tiles, trees)
    trees = remove_non_trees_safe(filtered_tree_path, trees)
    trees = group_rare_species(trees, species_col="baumgruppe",
                                threshold=rare_species_threshold)

    classes = sorted(trees["baumgruppe_grouped"].unique())
    label_map = {name: i + 1 for i, name in enumerate(classes)}  # 0 = Hintergrund
    num_classes = len(classes) + 1

    # WICHTIG: trees.copy() pro Kachel, da trees_geo_to_pixel das
    # uebergebene DataFrame in-place mutiert
    trees_by_tile = {}
    for tile_idx in range(len(tiles)):
        trees_for_tile = trees_geo_to_pixel(tiles, trees.copy(), tile_idx)
        trees_within = tree_selection(tiles, trees_for_tile, tile_idx)
        trees_by_tile[tile_idx] = trees_within

    patch_index = []
    for tile_idx in range(len(tiles)):
        tile_data = tiles[tile_idx]["data"]
        windows = generate_patch_windows(tile_data.width, tile_data.height,
                                          patch_size=patch_size, stride=patch_stride)
        for col_off, row_off in windows:
            patch_index.append((tile_idx, col_off, row_off))

    return tiles, trees_by_tile, patch_index, label_map, num_classes

#                                                            #
# #                                                        # #
# # # # # end build_dataset_components # # # # # # # # # # # #

# Modell

## Model: Funktionen

### Modell-Funktion: Daten-Split für Train/Val/Test

In [16]:
# ---------------------------------------------------------------------------
# 8. Train/Val/Test-Split (zufaellig ueber alle Patches, siehe Einschraenkung
#    zu moeglichem Data Leakage zwischen benachbarten Patches derselben Kachel)
# ---------------------------------------------------------------------------

def split_patch_index(patch_index, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, seed=RANDOM_SEED):
    rng = random.Random(seed)
    shuffled = patch_index.copy()
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    train_idx = shuffled[:n_train]
    val_idx = shuffled[n_train:n_train + n_val]
    test_idx = shuffled[n_train + n_val:]

    return train_idx, val_idx, test_idx

### Modell-Funktion: build_model

In [17]:
# ---------------------------------------------------------------------------
# 9. Modell aufbauen (Predictor-Kopf auf eigene Klassenzahl anpassen)
# ---------------------------------------------------------------------------

#import torch
#from torchvision.models.detection import fasterrcnn_resnet50_fpn
#from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
# -> bereits eingangs geladen

# # # # # # # # build_model # # # # # # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def build_model(num_classes):
    """
    Laedt Faster R-CNN mit vortrainiertem ResNet50-FPN-Backbone und
    ersetzt den Klassifikationskopf (urspruenglich 91 COCO-Klassen)
    durch einen auf num_classes (inkl. Hintergrund) zugeschnittenen Kopf.
    """
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

#                                                            #
# #                                                        # #
# # # # # end build_model # # # # # # # # # # # # # # # # # #

### Modell-Funktion: train_model

In [18]:
# ---------------------------------------------------------------------------
# 10. Trainingsloop
# ---------------------------------------------------------------------------

# # # # # # # # train_model # # # # # # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def train_model(model, train_loader, val_loader, num_epochs=5,
                 lr=0.005, device=None):
    """
    Einfacher Trainingsloop. Liefert nur die Trainings-Loss pro Epoche.

    HINWEIS: torchvision's Faster R-CNN gibt im train()-Modus nur Losses
    zurueck, keine Predictions. Fuer eine echte Val-Metrik (z.B. mAP)
    muesste man model.eval() nutzen und Predictions gegen targets
    auswerten (z.B. mit torchmetrics.detection.MeanAveragePrecision).
    Hier bewusst ausgelassen, um den Umfang fuer den didaktischen Zweck
    begrenzt zu halten.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Training auf Device: {device}")
    if device.type == "cpu":
        print(
            "WARNUNG: kein GPU erkannt - Training laeuft auf CPU und wird "
            "deutlich langsamer sein (ggf. mehrere Minuten pro Epoche statt "
            "Sekunden). In Colab: Laufzeit > Laufzeittyp aendern > GPU."
        )

    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=0.0005)

    loss_history = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()

            if batch_idx % 5 == 0:
                print(
                    f"  Epoch {epoch + 1}/{num_epochs} - "
                    f"Batch {batch_idx + 1}/{len(train_loader)} - "
                    f"loss: {losses.item():.4f} - device: {device}"
                )

        avg_loss = epoch_loss / max(len(train_loader), 1)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch + 1}/{num_epochs} - avg. training loss: {avg_loss:.4f}")

    return model, loss_history

#                                                            #
# #                                                        # #
# # # # # end train_model # # # # # # # # # # # # # # # # # #

## Modellierung durchführen

In [19]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "kein GPU")

True Tesla T4


In [37]:
# ---------------------------------------------------------------------------
# Datenaufbau
# ---------------------------------------------------------------------------

tiles, trees_by_tile, patch_index, label_map, num_classes = build_dataset_components(
    data_path=DATA_PATH,
    ntiles=NTILES,
    file_ext=FILE_EXT,
    filtered_tree_path=FILTERED_TREE_PATH,
    rare_species_threshold=RARE_SPECIES_THRESHOLD,
    patch_size=PATCH_SIZE,
    patch_stride=PATCH_STRIDE,
)

print(f"Gesamtzahl Patches: {len(patch_index)}")
print(f"Anzahl Klassen (inkl. Hintergrund): {num_classes}")

train_idx, val_idx, test_idx = split_patch_index(
    patch_index, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC
)
print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")


WARNUNG: 2 Zeilen mit doppeltem geom_key in trees_filtered nach CRS-Angleich (Rundung 1 Nachkommastellen). Betroffene Zeilen:
                    geom_key
29908  (407341.3, 5760859.7)
37496  (407341.3, 5760859.7)
WARNUNG: 2 Zeilen mit doppeltem geom_key in trees (Original-Baumdaten). Betroffene Zeilen:
                    geom_key
29908  (407341.3, 5760859.7)
37496  (407341.3, 5760859.7)
Gesamtzahl Patches: 729
Anzahl Klassen (inkl. Hintergrund): 26
Train: 510  Val: 109  Test: 110


In [38]:
# ---------------------------------------------------------------------------
# Datasets & Dataloader
# ---------------------------------------------------------------------------

train_ds = TreePatchDataset(tiles, trees_by_tile, train_idx, label_map,
                             patch_size=PATCH_SIZE, box_width=BOX_WIDTH)
val_ds = TreePatchDataset(tiles, trees_by_tile, val_idx, label_map,
                           patch_size=PATCH_SIZE, box_width=BOX_WIDTH)
test_ds = TreePatchDataset(tiles, trees_by_tile, test_idx, label_map,
                            patch_size=PATCH_SIZE, box_width=BOX_WIDTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn)


In [40]:
# ---------------------------------------------------------------------------
# Modell & Training
# ---------------------------------------------------------------------------

model = build_model(num_classes)
model, loss_history = train_model(model, train_loader, val_loader,
                                   num_epochs=NUM_EPOCHS, lr=LEARNING_RATE)

# Gewichte speichern, damit spaeter (z.B. bei Aenderungen an der Auswertung)
# nicht erneut trainiert werden muss. In Colab liegt das im ephemeren
# Dateisystem - bei Bedarf zusaetzlich nach Google Drive kopieren, damit
# es eine Session ueberdauert.
MODEL_WEIGHTS_PATH = "fasterrcnn_trees.pth"
torch.save(model.state_dict(), MODEL_WEIGHTS_PATH)
print(f"Modellgewichte gespeichert unter: {MODEL_WEIGHTS_PATH}")

# Optional, aber empfohlen: zusaetzlich nach Google Drive sichern, da das
# lokale Colab-Dateisystem mit der Session verloren geht (z.B. bei Timeout
# oder Laufzeittyp-Wechsel wie GPU<->CPU). GDRIVE_PROJECT_DIR bei Bedarf
# an die eigene Drive-Struktur anpassen. Sowohl Modellgewichte als auch
# alle Auswertungs-Outputs (Loss-Kurve, Testmetriken, Prediction-Bilder)
# landen unterhalb dieses einen Ordners.
GDRIVE_BACKUP = True
GDRIVE_PROJECT_DIR = "/content/drive/MyDrive/techlabs_trees_26"
GDRIVE_WEIGHTS_PATH = f"{GDRIVE_PROJECT_DIR}/fasterrcnn_trees.pth"
GDRIVE_RESULTS_DIR = f"{GDRIVE_PROJECT_DIR}/results"

if GDRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    from pathlib import Path as _Path
    _Path(GDRIVE_WEIGHTS_PATH).parent.mkdir(parents=True, exist_ok=True)
    _Path(GDRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)

    # Schutz gegen SameFileError: falls das aktuelle Arbeitsverzeichnis
    # bereits innerhalb von GDRIVE_PROJECT_DIR liegt (z.B. weil das
    # Notebook direkt aus Drive heraus arbeitet), zeigen MODEL_WEIGHTS_PATH
    # und GDRIVE_WEIGHTS_PATH bereits auf dieselbe physische Datei - dann
    # ist kein Kopieren noetig/moeglich.
    src_resolved = _Path(MODEL_WEIGHTS_PATH).resolve()
    dst_resolved = _Path(GDRIVE_WEIGHTS_PATH).resolve()

    if src_resolved == dst_resolved:
        print(
            f"Modellgewichte liegen bereits direkt in Drive "
            f"({GDRIVE_WEIGHTS_PATH}) - kein zusaetzliches Kopieren noetig."
        )
    else:
        shutil.copy(MODEL_WEIGHTS_PATH, GDRIVE_WEIGHTS_PATH)
        print(f"Zusaetzlich gesichert unter: {GDRIVE_WEIGHTS_PATH}")


# Zum spaeteren Laden, OHNE erneutes Training (z.B. in einer neuen Session,
# nachdem Drive erneut gemountet wurde):
#
   model = build_model(num_classes)
   model.load_state_dict(torch.load(GDRIVE_WEIGHTS_PATH))


IndentationError: unexpected indent (3347868394.py, line 2)

In [32]:
print(loss_history)

[0.9689043889201321, 0.72945699247375, 0.6789989172423307, 0.6770108087664157, 0.6498466844843552]


## Modell: Evaluation

### Modell-Evaluation: Funktionen

In [25]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches


# # # # # # # # plot_loss_curve # # # # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def plot_loss_curve(loss_history, output_path=None):
    """
    loss_history: Liste von avg. Training Loss je Epoche
                  (Rueckgabewert von train_model).
    """
    epochs = range(1, len(loss_history) + 1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, loss_history, marker="o")
    ax.set_xlabel("Epoche")
    ax.set_ylabel("Avg. Training Loss")
    ax.set_title("Trainingsverlauf")
    ax.grid(True, alpha=0.3)

    if output_path:
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"saved: {output_path}")

    plt.show()
    return fig

#                                                            #
# #                                                        # #
# # # # # end plot_loss_curve # # # # # # # # # # # # # # # #


# # # # # # # # evaluate_model (mAP) # # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def evaluate_model(model, data_loader, device=None):
    """
    Berechnet mAP (und Teilmetriken) auf einem Dataloader (z.B. Testset)
    mittels torchmetrics. Benoetigt vorherige Installation:
        pip install torchmetrics -q

    Rueckgabe: dict mit u.a. 'map', 'map_50', 'map_75'
    (torchmetrics-Tensoren, mit .item() in Zahlen umwandelbar).
    """
    from torchmetrics.detection.mean_ap import MeanAveragePrecision

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()

    metric = MeanAveragePrecision(box_format="xyxy")

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            preds = model(images)

            preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
            targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]

            metric.update(preds_cpu, targets_cpu)

    results = metric.compute()

    print("Testset-Metriken:")
    for key in ["map", "map_50", "map_75", "mar_1", "mar_10", "mar_100"]:
        if key in results:
            print(f"  {key}: {results[key].item():.4f}")

    return results

#                                                            #
# #                                                        # #
# # # # # end evaluate_model # # # # # # # # # # # # # # # # #


# # # # # # # # visualize_predictions # # # # # # # # # # # #
# #                                                        # #
#                                                            #
def visualize_predictions(model, dataset, indices, label_map, device=None,
                           score_threshold=0.5, output_dir=None):
    """
    Zeigt fuer ausgewaehlte Patch-Indices (Indices in `dataset`, NICHT
    tile-Indices) Ground-Truth-Boxen (gruen) und Modell-Predictions
    oberhalb von score_threshold (rot, mit Klassenname + Score) im
    selben Bild an.

    label_map: dict Artname -> label_id (wie beim Dataset-Aufbau
               verwendet). Wird intern invertiert.
    """
    inv_label_map = {v: k for k, v in label_map.items()}
    inv_label_map[0] = "Hintergrund"

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()

    for i in indices:
        image, target = dataset[i]

        with torch.no_grad():
            pred = model([image.to(device)])[0]

        img_np = np.transpose(image.numpy(), (1, 2, 0))

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(img_np)
        ax.axis("off")

        # Ground Truth: gruen
        for box, label in zip(target["boxes"], target["labels"]):
            xmin, ymin, xmax, ymax = box.tolist()
            rect = patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                fill=False, edgecolor="lime", linewidth=1.5
            )
            ax.add_patch(rect)
            ax.text(
                xmin, ymax + 3, inv_label_map.get(int(label), str(int(label))),
                color="lime", fontsize=7,
                bbox=dict(facecolor="black", alpha=0.5, pad=0.5)
            )

        # Predictions: rot, nur oberhalb Schwellenwert
        keep = pred["scores"] >= score_threshold
        boxes = pred["boxes"][keep].cpu()
        labels = pred["labels"][keep].cpu()
        scores = pred["scores"][keep].cpu()

        for box, label, score in zip(boxes, labels, scores):
            xmin, ymin, xmax, ymax = box.tolist()
            rect = patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                fill=False, edgecolor="red", linewidth=1.5
            )
            ax.add_patch(rect)
            label_name = inv_label_map.get(int(label), str(int(label)))
            ax.text(
                xmin, ymin - 3, f"{label_name} {score:.2f}",
                color="red", fontsize=7,
                bbox=dict(facecolor="white", alpha=0.6, pad=0.5)
            )

        ax.set_title(
            f"Patch idx {i} - gruen: Ground Truth, rot: Prediction "
            f"(Score >= {score_threshold})"
        )

        if output_dir:
            out_path = f"{output_dir}/prediction_patch_{i}.png"
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            print(f"saved: {out_path}")

        plt.show()
        plt.close(fig)

#                                                            #
# #                                                        # #
# # # # # end visualize_predictions # # # # # # # # # # # # #

### Modell-Evaluation: Durchführen
Es wird die Lossfunktion über die Trainings-Epochen als zentrale Metrik ausgegeben. Außerdem werden weitere Testset-Metriken ausgegeben. Schließlich werden exemplarisch Luftbilder mit korrekt bzw. nicht korrekt Klassifizierten Bäumen zur visuellen Inspektion ausgegeben.
Der Output wird (Laufzeit Googel Colab) in der Subfolder 'results' gespeichert.

In [35]:
# ---------------------------------------------------------------------------
# Auswertung: Loss-Kurve, mAP auf Testset, visuelle Kontrolle
# Alle Outputs werden direkt nach Drive geschrieben (GDRIVE_RESULTS_DIR),
# sofern GDRIVE_BACKUP = True. Andernfalls landen sie nur lokal im
# ephemeren Colab-Dateisystem.
# ---------------------------------------------------------------------------

results_dir = GDRIVE_RESULTS_DIR if GDRIVE_BACKUP else "."

plot_loss_curve(loss_history, output_path=f"{results_dir}/loss_curve.png")

# Benoetigt: pip install torchmetrics -q  (einmalig in Colab ausfuehren)
test_metrics = evaluate_model(model, test_loader)

# Testmetriken als JSON sichern, damit sie fuer die Praesentation ohne
# erneuten Testlauf verfuegbar sind. torchmetrics liefert Tensoren -
# .tolist() wandelt sowohl Skalare als auch Listen in JSON-taugliche
# Python-Typen um.
import json

test_metrics_serializable = {
    k: (v.tolist() if hasattr(v, "tolist") else v)
    for k, v in test_metrics.items()
}
metrics_path = f"{results_dir}/test_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(test_metrics_serializable, f, indent=2)
print(f"Testmetriken gespeichert unter: {metrics_path}")

# ein paar Beispiel-Patches aus dem Testset visuell pruefen
# (Indices beziehen sich auf test_ds, nicht auf test_idx/tile-Indices)
n_examples = min(5, len(test_ds))
example_indices = list(range(n_examples))
visualize_predictions(model, test_ds, example_indices, label_map,
                       score_threshold=0.5, output_dir=results_dir)

Output hidden; open in https://colab.research.google.com to view.